In [ ]:
# !pip uninstall -y numpy matplotlib scikit-learn imbalanced-learn
# !pip install numpy==1.23.5 matplotlib==3.6.3 scikit-learn==1.1.3 imbalanced-learn==0.10.1

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: matplotlib 3.7.1
Uninstalling matplotlib-3.7.1:
  Successfully uninstalled matplotlib-3.7.1
Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2
Found existing installation: imbalanced-learn 0.12.3
Uninstalling imbalanced-learn-0.12.3:
  Successfully uninstalled imbalanced-learn-0.12.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.5/30.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.0/226.0 kB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore

In [ ]:
import os
import time
import pickle

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fftpack import dct

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE

In [ ]:

# Replace 'your_file.csv' with the actual path to your CSV file
file_path = r'/kaggle/input/datasets/saurabhsaini1/auitd-dataset/Labels_AUITD.csv'

column_names = ['Image', 'Stage_1', 'Stage_2']

df = pd.read_csv(file_path, header=None, names=column_names)

df=df[1:792]

print(df)

     Image Stage_1 Stage_2
1    2_100       1       2
2    2_101       1       2
3    2_102       1       2
4    2_103       1       2
5    2_104       1       2
..     ...     ...     ...
787   5_95       2       5
788   5_96       2       5
789   5_97       2       5
790   5_98       2       5
791   5_99       2       5

[791 rows x 3 columns]


In [ ]:


# Define the root directory
rootdir = r'/kaggle/input/datasets/saurabhsaini1/auitd-dataset/All_images/All_images'
image_path = [os.path.join(rootdir, file) for file in os.listdir(rootdir)]

# Function to generate zigzag order indices
def zigzag_indices(n):
    index_order = sorted(((x, y) for x in range(n) for y in range(n)),
                         key=lambda s: (s[0] + s[1], s[1] if (s[0] + s[1]) % 2 else -s[1]))
    return [(x, y) for x, y in index_order]

# Function to extract localized DCT descriptors
def extract_localized_dct_descriptors(image):
    # Resize image and convert to grayscale
    resized_image = cv2.resize(image, (256, 256))
    gray_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)
    height, width = gray_image.shape
    descriptors = []
    zigzag_order = zigzag_indices(8)

    # Apply localized DCT in 8x8 blocks
    for i in range(0, height, 8):
        for j in range(0, width, 8):
            cell = gray_image[i:i + 8, j:j + 8]
            
            # Apply DCT
            dct_cell = cv2.dct(np.float32(cell))
            
            # Keep first 36 coefficients in zigzag order
            cell_descriptor = [dct_cell[idx] for idx in zigzag_order[:36]]
            
            # Compute average
            avg_value = np.mean(cell_descriptor)
            
            # Binary thresholding
            binary_descriptor = [1 if coeff > avg_value else 0 for coeff in cell_descriptor]
            
            # Convert binary to decimal
            decimal_value = int("".join(map(str, binary_descriptor)), 2)
            
            descriptors.append(decimal_value)

    return np.array(descriptors)


# Define a global dataframe (UPDATED)
dct_df = pd.DataFrame(columns=['image_name', 'features', 'labels_1', 'labels_2'])

def main():
    global dct_df
    t0 = time.time()

    for file in image_path:
        img = cv2.imread(file)
        
        # Extract image name
        img_name = os.path.basename(file).split('.')[0]
        
        # Extract labels from df (assumes df is already defined)
        label_1 = df.loc[df['Image'] == img_name, 'Stage_1'].iloc[0]
        label_2 = df.loc[df['Image'] == img_name, 'Stage_2'].iloc[0]
        
        # Apply localized DCT
        img_des = extract_localized_dct_descriptors(img)
        
        if img_des is not None:
            new_data = pd.DataFrame({
                'image_name': [img_name],   # ✅ Added this line
                'features': [img_des],
                'labels_1': [label_1],
                'labels_2': [label_2]
            })
            
            dct_df = pd.concat([dct_df, new_data], ignore_index=True)
    
    print('Time taken: {:.2f} seconds'.format(time.time() - t0))


print('\nCalculating')
main()

print(dct_df)


Calculating
Time taken: 35.51 seconds
    image_name                                           features labels_1  \
0         5_59  [42163597256, 48434031880, 48597894400, 486481...        2   
1        4_316  [35037381120, 54846161152, 39539180160, 523491...        2   
2        4_122  [41503380164, 47882182790, 47861399765, 499296...        2   
3        4_388  [58406146588, 42132055119, 42229157445, 400605...        2   
4         4_77  [45787256445, 37067539036, 37151998024, 349619...        2   
..         ...                                                ...      ...   
776       3_87  [38271266434, 35176579072, 35169239296, 344965...        1   
777       2_87  [43169616260, 36722182568, 45174358472, 472500...        1   
778       3_62  [55566191632, 35254174976, 50065309952, 477846...        1   
779       5_80  [37475266626, 34903949376, 47823060992, 394631...        2   
780       2_77  [37207301837, 47846811852, 43568478432, 347267...        1   

    labels_2  
0        

In [5]:
print("Length of features in hog df", len(dct_df['features'].iloc[-1]))

Length of features in hog df 1024


In [6]:
dct_df

,image_name,features,labels_1,labels_2
0,5_59,"[42163597256, 48434031880, 48597894400, 486481...",2,5
1,4_316,"[35037381120, 54846161152, 39539180160, 523491...",2,4
2,4_122,"[41503380164, 47882182790, 47861399765, 499296...",2,4
3,4_388,"[58406146588, 42132055119, 42229157445, 400605...",2,4
4,4_77,"[45787256445, 37067539036, 37151998024, 349619...",2,4
...,...,...,...,...
776,3_87,"[38271266434, 35176579072, 35169239296, 344965...",1,3
777,2_87,"[43169616260, 36722182568, 45174358472, 472500...",1,2
778,3_62,"[55566191632, 35254174976, 50065309952, 477846...",1,3
779,5_80,"[37475266626, 34903949376, 47823060992, 394631...",2,5


In [7]:

# Dropping the 'labels_1' column
dct_df_stage_1 = dct_df.drop(columns=['labels_2'])
# Dropping the 'labels_1' column
dct_df_stage_2 = dct_df.drop(columns=['labels_1'])



In [8]:
# Display the updated DataFrame
print(dct_df_stage_1)

    image_name                                           features labels_1
0         5_59  [42163597256, 48434031880, 48597894400, 486481...        2
1        4_316  [35037381120, 54846161152, 39539180160, 523491...        2
2        4_122  [41503380164, 47882182790, 47861399765, 499296...        2
3        4_388  [58406146588, 42132055119, 42229157445, 400605...        2
4         4_77  [45787256445, 37067539036, 37151998024, 349619...        2
..         ...                                                ...      ...
776       3_87  [38271266434, 35176579072, 35169239296, 344965...        1
777       2_87  [43169616260, 36722182568, 45174358472, 472500...        1
778       3_62  [55566191632, 35254174976, 50065309952, 477846...        1
779       5_80  [37475266626, 34903949376, 47823060992, 394631...        2
780       2_77  [37207301837, 47846811852, 43568478432, 347267...        1

[781 rows x 3 columns]


In [9]:
# Display the updated DataFrame
print(dct_df_stage_2)

    image_name                                           features labels_2
0         5_59  [42163597256, 48434031880, 48597894400, 486481...        5
1        4_316  [35037381120, 54846161152, 39539180160, 523491...        4
2        4_122  [41503380164, 47882182790, 47861399765, 499296...        4
3        4_388  [58406146588, 42132055119, 42229157445, 400605...        4
4         4_77  [45787256445, 37067539036, 37151998024, 349619...        4
..         ...                                                ...      ...
776       3_87  [38271266434, 35176579072, 35169239296, 344965...        3
777       2_87  [43169616260, 36722182568, 45174358472, 472500...        2
778       3_62  [55566191632, 35254174976, 50065309952, 477846...        3
779       5_80  [37475266626, 34903949376, 47823060992, 394631...        5
780       2_77  [37207301837, 47846811852, 43568478432, 347267...        2

[781 rows x 3 columns]


In [ ]:

# Assuming dct_df_stage_2 is already defined and contains the columns 'features' and 'labels_2'

# Convert labels_2 to numeric if it's not already
dct_df_stage_2['labels_2'] = pd.to_numeric(dct_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 2 or 3
dct_df_benign = dct_df_stage_2[dct_df_stage_2['labels_2'].isin([2, 3])]

# Display the filtered dataframe
print(dct_df_benign)


    image_name                                           features  labels_2
5         3_95  [55717136392, 56912511488, 52684652608, 526133...         3
9         2_57  [45787782900, 40872237184, 39487374008, 434847...         2
14        3_77  [52344913920, 51808045056, 51541704768, 612032...         3
16        3_42  [45508353467, 40807439824, 42982183376, 430873...         3
17        3_43  [39843943216, 57044633920, 53085208576, 551655...         3
..         ...                                                ...       ...
775      2_110  [53822021197, 48373077040, 48604460039, 531211...         2
776       3_87  [38271266434, 35176579072, 35169239296, 344965...         3
777       2_87  [43169616260, 36722182568, 45174358472, 472500...         2
778       3_62  [55566191632, 35254174976, 50065309952, 477846...         3
780       2_77  [37207301837, 47846811852, 43568478432, 347267...         2

[182 rows x 3 columns]


In [11]:
import pandas as pd

# Assuming dct_df_stage_2 is already defined and contains the columns 'features' and 'labels_2'

# Convert labels_2 to numeric if it's not already
dct_df_stage_2['labels_2'] = pd.to_numeric(dct_df_stage_2['labels_2'])

# Filter the dataframe to include only rows with labels_2 values 4 or 5
dct_df_malignant = dct_df_stage_2[dct_df_stage_2['labels_2'].isin([4,5])]

# Display the filtered dataframe
print(dct_df_malignant)


    image_name                                           features  labels_2
0         5_59  [42163597256, 48434031880, 48597894400, 486481...         5
1        4_316  [35037381120, 54846161152, 39539180160, 523491...         4
2        4_122  [41503380164, 47882182790, 47861399765, 499296...         4
3        4_388  [58406146588, 42132055119, 42229157445, 400605...         4
4         4_77  [45787256445, 37067539036, 37151998024, 349619...         4
..         ...                                                ...       ...
767       4_13  [41497088596, 47362154662, 49176660364, 436405...         4
769      4_493  [67067383808, 57117913984, 52914815360, 379863...         4
771      4_156  [40002362563, 48554346122, 37737538176, 400421...         4
772      4_248  [36588028352, 34629879200, 43257168368, 367130...         4
779       5_80  [37475266626, 34903949376, 47823060992, 394631...         5

[599 rows x 3 columns]


# Stage2 Malignant(TIRADS4/TIRADS5)

In [ ]:
# =====================================================
# Ensure labels are int
# =====================================================

dct_df_malignant['labels_2'] = (
    dct_df_malignant['labels_2'].astype(int)
)

# =====================================================
# Separate features, labels, names
# =====================================================

X = np.array(
    dct_df_malignant['features'].tolist()
)

y = np.array(
    dct_df_malignant['labels_2']
)

names = np.array(
    dct_df_malignant['image_name']
)

# =====================================================
# Kaggle output root
# =====================================================

output_root = "/kaggle/working"

model_dir = os.path.join(
    output_root,
    "models_stage_2"
)

result_dir = os.path.join(
    output_root,
    "results_stage_2"
)

os.makedirs(model_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

# =====================================================
# Parameters
# =====================================================

num_splits = 5
kernel = 'rbf'

# =====================================================
# Metric storage
# =====================================================

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

# =====================================================
# Stratified KFold
# =====================================================

kfold = StratifiedKFold(
    n_splits=num_splits,
    shuffle=True
)

# =====================================================
# Store predictions
# =====================================================

all_results = pd.DataFrame(
    columns=[
        'fold',
        'image_name',
        'true_label',
        'predicted_label'
    ]
)

# =====================================================
# Cross-validation
# =====================================================

for fold, (train_index, test_index) in enumerate(
        kfold.split(X, y), start=1):

    print("\n" + "="*50)
    print(f"Fold {fold}")
    print("="*50)

    # =================================================
    # Split data
    # =================================================

    X_train = X[train_index]
    X_test  = X[test_index]

    y_train = y[train_index]
    y_test  = y[test_index]

    names_test = names[test_index]

    # =================================================
    # Original Test Distribution
    # =================================================

    unique_test, counts_test = np.unique(
        y_test,
        return_counts=True
    )

    print("\nOriginal Test Distribution:")

    for cls, cnt in zip(unique_test, counts_test):
        print(f"Class {cls}: {cnt}")

    print(f"\nTotal Test Samples: {len(y_test)}")

    # =================================================
    # SMOTE on TRAIN
    # =================================================

    smote_train = SMOTE(random_state=5)

    X_train_smote, y_train_smote = (
        smote_train.fit_resample(
            X_train,
            y_train
        )
    )

    # =================================================
    # Training Distribution After SMOTE
    # =================================================

    print("\nTraining Distribution After SMOTE:")

    unique_sm, counts_sm = np.unique(
        y_train_smote,
        return_counts=True
    )

    for cls, cnt in zip(unique_sm, counts_sm):
        print(f"Class {cls}: {cnt}")

    X_test_final = X_test
    y_test_final = y_test

    # =================================================
    # TRAIN SVM
    # =================================================

    svm = SVC(
        kernel=kernel
    
    )

    svm.fit(
        X_train_smote,
        y_train_smote
    )

    # =================================================
    # PREDICT
    # =================================================

    y_pred = svm.predict(
        X_test_final
    )

    # =================================================
    # Save model
    # =================================================

    model_path = os.path.join(
        model_dir,
        f"svm_fold_{fold}.pkl"
    )

    with open(model_path, 'wb') as f:
        pickle.dump(svm, f)

    # =================================================
    # Save fold predictions
    # =================================================

    fold_df = pd.DataFrame({
        'fold': [fold] * len(y_test_final),
        'image_name': ['mixed_test']
                      * len(y_test_final),
        'true_label': y_test_final,
        'predicted_label': y_pred
    })

    all_results = pd.concat(
        [all_results, fold_df],
        ignore_index=True
    )

    # =================================================
    # Confusion Matrix
    # =================================================

    cm = confusion_matrix(
        y_test_final,
        y_pred,
    )

    tn, fp, fn, tp = cm.ravel()

    # =================================================
    # Metrics
    # =================================================

    specificity = (
        tn / (tn + fp + 1e-10)
    )

    sensitivity = (
        tp / (tp + fn + 1e-10)
    )

    accuracy = (
        (tp + tn)
        / (tp + tn + fp + fn + 1e-10)
    )

    precision = (
        tp / (tp + fp + 1e-10)
    )

    f1_score = (
        2 * (precision * sensitivity)
    ) / (
        precision + sensitivity + 1e-10
    )

    # =================================================
    # Store metrics
    # =================================================

    specificity_scores.append(
        specificity
    )

    sensitivity_scores.append(
        sensitivity
    )

    accuracy_scores.append(
        accuracy
    )

    precision_scores.append(
        precision
    )

    f1_scores.append(
        f1_score
    )

# =====================================================
# SAVE CSV
# =====================================================

csv_path = os.path.join(
    result_dir,
    "fold_predictions.csv"
)

all_results.to_csv(
    csv_path,
    index=False
)

# =====================================================
# Average Metrics
# =====================================================

average_accuracy = np.mean(
    accuracy_scores
)

average_specificity = np.mean(
    specificity_scores
)

average_sensitivity = np.mean(
    sensitivity_scores
)

average_precision = np.mean(
    precision_scores
)

average_f1_score = np.mean(
    f1_scores
)

# =====================================================
# FINAL RESULTS
# =====================================================

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print("\nAverage Precision:",
      average_precision)

print("Average F1 Score:",
      average_f1_score)

print("\nAverage Specificity:",
      average_specificity)

print("Average Sensitivity (Recall):",
      average_sensitivity)

print("Average Accuracy:",
      average_accuracy)


Fold 1

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120

Training Distribution After SMOTE:
Class 4: 404
Class 5: 404


<ipython-input-13-296134346abf>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dct_df_malignant['labels_2'] = (



Fold 2

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120

Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 3

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120

Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 4

Original Test Distribution:
Class 4: 101
Class 5: 19

Total Test Samples: 120

Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

Fold 5

Original Test Distribution:
Class 4: 101
Class 5: 18

Total Test Samples: 119

Training Distribution After SMOTE:
Class 4: 404
Class 5: 404

FINAL RESULTS

Average Precision: 0.999999999998771
Average F1 Score: 0.8924413765687419

Average Specificity: 0.9999999999990099
Average Sensitivity (Recall): 0.805940594058608
Average Accuracy: 0.9029702970292561
